# Accessing Claude with the API

In [1]:
%pip install anthropic

In [2]:
from google.colab import userdata
api_key = userdata.get('ANTHROPIC_API_KEY').strip()

In [3]:
from anthropic import Anthropic
client = Anthropic(api_key=api_key)
model = "claude-sonnet-4-0"

In [4]:
# helper functions
def add_user_message(messages, text):
  user_message = {'role': 'user', 'content': text}
  messages.append(user_message)

def add_assistant_message(messages, text):
  assistant_message = {'role': 'assistant', 'content': text}
  messages.append(assistant_message)

def chat(messages):
  message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages
  )

  return message.content[0].text

In [ ]:
# multi-turn conversation
messages = []
add_user_message(messages, 'What can I do with the Claude API? Please answer in one sentence.')
answer = chat(messages)
add_assistant_message(messages, answer)
add_user_message(messages, 'Write one more sentence')
answer = chat(messages)
answer

'The API supports both synchronous and streaming responses, allowing you to build chatbots, content generation tools, automated writing assistants, and other AI-powered applications that can process and respond to text input in real-time.'

In [ ]:
# mini chatbot
messages = []

while True:
  user_input = input('> ')
  if user_input == 'stop':
    break

  add_user_message(messages, user_input)
  answer = chat(messages)
  add_assistant_message(messages, answer)
  print(f'Chatbot: {answer}')


> How's it going?
Chatbot: Going well, thanks for asking! I'm here and ready to help with whatever you'd like to chat about or work on. How are things with you today?
> I'm doing great as well. What should I do today? 
Chatbot: That's great to hear! To give you some good suggestions, it would help to know a bit more about your situation. A few questions:

- What kind of day is it for you - weekend, day off, or do you have work/school?
- Are you looking for something productive, relaxing, creative, or social?
- What's the weather like where you are?
- Any particular interests or things you've been wanting to try?

But here are some ideas that work for most any day:
- Take a walk outside or do some exercise
- Try cooking something new
- Read a book or listen to a podcast
- Call someone you haven't talked to in a while
- Work on a hobby or creative project
- Organize/declutter a space in your home
- Learn something new online

What sounds appealing to you, or what kind of vibe are you goi

In [6]:
# math tutor w/ system prompts
def chat(messages):
  system = """
  You are a patient math tutor.
  Do not directly answer a student's questions.
  Guide them to a solution step by step.
  """
  message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    system=system
  )

  return message.content[0].text

In [10]:
messages = []
add_user_message(messages, "How do I solve 2x+9=5x+2 for x?")
answer = chat(messages)
answer

"Great question! Let's work through this step by step. \n\nFirst, let me ask you: what do you think our goal should be when solving for x? What would we want the equation to look like at the end?\n\nAnd as a starting point, can you tell me what you see on the left side of the equation and what you see on the right side?"

In [12]:
# flexible system prompts
def chat(messages, system=None):
  params = {
    'model':model,
    'max_tokens':1000,
    'messages':messages
  }
  if system:
    params['system'] = system

  message = client.messages.create(**params)
  return message.content[0].text

In [17]:
messages = []

system = """
You are a high-level coding assistant that writes precise Python code.
Make sure that the response is short and sweet.
"""

add_user_message(messages, 'Write a python function that returns the maximum value in a list.')
answer = chat(messages, system)
answer

'```python\ndef max_value(lst):\n    return max(lst)\n```\n\nOr if you need to handle empty lists:\n\n```python\ndef max_value(lst):\n    return max(lst) if lst else None\n```'

In [18]:
# temperature
def chat(messages, system=None, temperature=1.0):
  params = {
    'model':model,
    'max_tokens':1000,
    'messages':messages,
    'temperature':temperature
  }
  if system:
    params['system'] = system

  message = client.messages.create(**params)
  return message.content[0].text

In [30]:
messages = []

add_user_message(messages, 'Give me a one sentence movie recommendation.')
answer = chat(messages, temperature=0.0)
# the movie will likely be 'The Grand Budapest Hotel'
answer

'"The Grand Budapest Hotel" is a visually stunning and whimsically charming comedy that follows the adventures of a legendary concierge and his protégé at a famous European hotel, blending Wes Anderson\'s distinctive style with a delightfully eccentric murder mystery.'

In [31]:
messages = []

add_user_message(messages, 'Give me a one sentence movie recommendation.')
answer = chat(messages, temperature=1.0)
# more random and unique suggestions
answer

'Watch "Parasite" (2019) for a masterfully crafted thriller that seamlessly blends dark comedy, social commentary, and suspense into an unforgettable cinematic experience.'

In [35]:
# response streaming
messages = []

add_user_message(messages, 'Write a one sentence description of a fake database.')

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
  )

for event in stream:
  print(event)

RawMessageStartEvent(message=Message(id='msg_013vGjW3kLCzgC3KnjjFja1J', container=None, content=[], model='claude-sonnet-4-20250514', role='assistant', stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=17, output_tokens=1, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='The', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' "GlobalPetPreferences" database contains comprehensive records', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' of pet ownership statistics, behavioral pat

In [38]:
# extract just content from response stream
messages = []

add_user_message(messages, 'Write a one sentence description of a fake database.')

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
      # print(text, end="")
      pass

final_message = stream.get_final_message()

This database contains comprehensive records of all interdimensional pizza delivery routes used by the Galactic Food Service between 2847 and 3012 CE.

In [40]:
# updated function w/ stop sequences
def chat(messages, system=None, temperature=1.0, stop_sequences=None):
  params = {
    'model':model,
    'max_tokens':1000,
    'messages':messages,
    'temperature':temperature
  }
  if system:
    params['system'] = system

  if stop_sequences:
    params['stop_sequences'] = stop_sequences

  message = client.messages.create(**params)
  return message.content[0].text

In [43]:
# structured data generation
messages = []

add_user_message(messages, 'Generate a very short event bridge rule as JSON.')
add_assistant_message(messages, "```json")
text = chat(messages, stop_sequences=['```'])
text

'\n{\n  "Name": "OrderProcessingRule",\n  "EventPattern": {\n    "source": ["myapp.orders"],\n    "detail-type": ["Order Placed"]\n  },\n  "State": "ENABLED",\n  "Targets": [\n    {\n      "Id": "1",\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:ProcessOrder"\n    }\n  ]\n}\n'

In [45]:
import json

json.loads(text.strip())

{'Name': 'OrderProcessingRule',
 'EventPattern': {'source': ['myapp.orders'], 'detail-type': ['Order Placed']},
 'State': 'ENABLED',
 'Targets': [{'Id': '1',
   'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:ProcessOrder'}]}

In [55]:
# prefilling and stop sequence
messages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""

add_user_message(messages, prompt)
add_assistant_message(messages, 'Here are all three commands in a single block without any comments:\n```bash')
text = chat(messages, stop_sequences=['```'])
text.strip()

'aws s3 ls\n\naws ec2 describe-instances\n\naws iam list-users'